# SIMT 算子调试

SIMT 的 bug 有鲜明的模式。本节给出高频错误清单、排查手段与能力边界。


## 1. 高频错误模式 Top5

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">#</th><th style="text-align:left;">症状</th><th style="text-align:left;">根因</th><th style="text-align:left;">预防</th></tr>
<tr><td>1</td><td>输出部分为 0 / 破坏相邻数据</td><td><b>动态下标越界</b>（前端不检查）</td><td>读写前显式 <code>0 ≤ idx &lt; Shape/valid_shape</code></td></tr>
<tr><td>2</td><td>计数/累加结果偏小</td><td><b>非原子读改写竞争</b></td><td>atomic_add/max</td></tr>
<tr><td>3</td><td>卡死（Kernel 不返回）</td><td><b>屏障在分支内</b>，部分线程未到达</td><td>屏障无条件下达；核对每条路径</td></tr>
<tr><td>4</td><td>结果随机错 / 偶发错</td><td>共享 Tile 的<b>跨线程依赖未同步</b></td><td>写→syncthreads→读</td></tr>
<tr><td>5</td><td>尾部数据未处理</td><td>线程数 < 数据量且未用 grid-stride</td><td>默认 grid-stride + 循环边界</td></tr>
</table>


## 2. 排查手段

1. **缩小规模**：1 Block × 1 线程跑通 → 1 Block × 256 → 多 Block——把"并发问题"退化成"顺序问题"定位；
2. **记录标量**：将目标线程的索引/中间值写到专门的 GM 调试 Tensor，Host 同步后读取；每个记录位置只允许一个线程写，不假定普通 printf 可进入 SIMT 域；
3. **对照 SIMD 版**：同算子写一个 Tile 版（小规模），逐块比对输出，定位错误线程对应的行/列；
4. **sanitizer**（第 8 章展开）：检查外层 SIMD Kernel 的访问和声明；当前不插桩 SIMT 线程块内的 GM/Tile 访问，因此无报告不能证明 SIMT 安全。


## 3. 案例实战（理论推导版）

**案例：Gather 输出"前段正确、后段全零"**

- **现象**：[1, 10000] 的 Gather，前 2048 个输出正确，其后全为零（或保持初值）；
- **定位思路**：部分正确 → 索引映射与边界判断基本正确 → "恰好 2048 = 8 核 × 256 线程" → 疑似**线程数 < 数据量且无跨步循环**（Top5 第 5 类）；
- **根因**：每线程只处理 `index < count` 的一个元素，超出全局线程总数的数据无人认领；
- **修复**：改 grid-stride（`for row in pl.range(first, count, stride)`）后回归全量 shape；
- **验证方法**：单线程版（`max_threads=1`、单核）做 golden 对照（第 7 章排查三板斧之一），第一个不匹配位置即线程覆盖边界。

【待 950 验证：案例为课程构造（依据第 7 章边界规则推导），可复现】


## 4. 当前能力边界（重要）

写 SIMT 前先对照这份清单，**不在清单内的能力一律不要假设**：

- 本课程的 SIMT VF 入口由外层 Vector 域调用，不支持嵌套启动入口；当前源码另有 Host 直接启动 SIMT Kernel，参见 01.05，不能将本课程路径限制扩大为全部能力限制；
- SIMT 内**不支持**：Tile 创建、SIMD Tile 计算、Reg 计算、System 流水操作；
- **不支持**：动态 GM Shape、Tile Subview、L1 Buffer Tile、DN/NZ 布局、通用指针参数；
- **版本边界**：当前源码 `_simt_api.py` 已提供 warp_shfl / warp_ballot / warp_reduce_*，并有解析单测；目标 CANN 9.2.0 / PyPTO 0.3.0 包是否含这些接口仍需核对，本课程基础实现不依赖它们。当前指南仍列线程私有数组、显式 Cached GM 访问为未提供。

遇到边界时的出路：数据准备/搬运移到外层 Kernel（Tile 侧做）；动态 Shape 用静态 Shape + count 参数模拟；不依赖新增 Warp API 的基础实现可用块内协作 + 原子表达；本质上需要 SIMD 批量的计算——回第 3 章，或等第 9 章协同。


### 让输入携带来源行号

调试 Gather 时可设 `data[row,col]=row*64+col`，索引使用 [0,1023,0,17,…]。输出不仅能显示“数值不对”，还可反推出实际读了哪一行。先比较所有输出行的首列，再检查列内连续性；重复索引应重复读同一行，合法边界 0 与 1023 应准确覆盖。

输出预填 NaN 后，连续一段 NaN 通常提示漏覆盖；有限但来自错误行的数值提示索引映射错误。这些是定位线索，需要与源码共同核对，不能仅凭一种输出形态断言根因。测试负索引时应先由 Host 拒绝，不能把主动越界访问当作正常验收。


## 5. 课后练习

1. （选择）"输出前 1000 个元素正确、之后全 0"最可能的原因：
   A. 原子操作缺失<br/>
   B. 线程数 < 数据量且无 grid-stride<br/>
   C. 屏障死锁
2. （判断）动态 GM Shape 在 SIMT 中可用运行时下标模拟。（对/错）
3. （实践）设计一个"1 线程版"Gather（max_threads=1、单核启动）作为调试基准：它能排除哪些并发因素，又仍需验证哪些索引/边界问题？用它如何定位多线程版的越界行？

In [ ]:
!cat ./answer/07.04_answer.txt

## 6. 本节小结

- Top5 错误模式：**越界、竞争、屏障分支、跨线程依赖、无跨步**；
- 排查三板斧：**缩规模、记录调试 Tensor、对照独立 Golden** + sanitizer；
- 能力边界清单：静态形状与 UB ND 仍是混合路径边界；Host SIMT/Warp API 需按版本核对——**先查边界再写代码**。